# Run: tiny_cnn + MNIST, 20 rounds, checkpointed

One notebook, run top to bottom, single Colab session (free tier).
This is the ONLY notebook you need for this run -- it's a consolidated,
linear version of `00_Setup_and_Crypto_Tests.ipynb` + `CASE1_IID_MNIST.ipynb`
with the smoke test and the 2nd backbone removed, so there's nothing to
skip or get out of order.

**Scope** (see `FIX.md`/`STATUS.md`, 2026-11-15 entries, for the full
reasoning): 1 model -- `tiny_cnn` (~20.6K params), NOT resnet18 --
matching the model scale used by this paper's actual peer group
(WhiteCloak NDSS'26, PQS-BFL 2025, both ~19-22K params on MNIST), on
MNIST, `n_clients=50` (all 50 go through the crypto pipeline every
round -- `sampling_rate` only gates real-vs-zero local training, not
crypto participation), 20 rounds. Real measured per-round cost with
resnet18 was ~72 min/round (~24h for 20 rounds) because resnet18 is
~500x bigger than every comparable paper's model; tiny_cnn should cut
that dramatically since mask/sketch/range-proof cost is O(model_dim).

**Checkpoint durability: git, not Drive (2026-11-16).** Drive isn't
mounted in this notebook -- checkpoints are committed and pushed, after
every round, to a dedicated `colab-checkpoints-case1-tiny_cnn` branch on the FL-ZKP
repo itself -- unique to this notebook, never shared with the other
MVP run notebooks (FIX.md F17) -- never `main` (see the git-checkpoint-setup cell's own
markdown for why a separate branch, not Drive, not a per-round commit on
`main`). A push failing (transient network blip) only costs a retry
next round -- it does not stop training, and the checkpoint is always
safe on local disk regardless.

**If Colab disconnects partway through:** re-run from cell 1 again.
The repo stays cloned across a disconnect/reconnect; the git-checkpoint-
setup cell re-clones the dedicated checkpoint checkout fresh (that one
does NOT survive a disconnect, only what's pushed to GitHub does) and
fetches whatever was last pushed to `colab-checkpoints-case1-tiny_cnn` -- the run cell
then finds that checkpoint and resumes after the last completed round,
it does not restart from round 0. If you get a `NameError` for a
variable partway through re-running, it just means you haven't reached
that cell yet this pass -- keep running forward, don't skip around.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).

In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())

In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")

In [ ]:
# 7. Build the model.
# tiny_cnn (~20.6K params), not resnet18: matches the model scale used
# by this paper's actual peer group (WhiteCloak NDSS'26: 19K-param CNN
# on MNIST; PQS-BFL 2025: ~21.8K-param CNN on MNIST) -- see
# fl/models.py::build_tiny_cnn's docstring for the full citation
# reasoning. ~500x fewer params than resnet18's 11.2M, which is most of
# why this run is dramatically cheaper per round.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability via git (not Drive)

**Why a separate clone, a separate branch, and a callback -- not just
`git commit` inside `/content/FL-ZKP`:**
- A SEPARATE clone (`/content/FL-ZKP-checkpoints`) so committing a
  checkpoint can never touch the code checkout this notebook is actually
  importing from -- no risk of a `git checkout`/`reset` on the
  checkpoint side disturbing `sys.path`-imported modules mid-run.
- A dedicated `colab-checkpoints-case1-tiny_cnn` branch, never `main`, unique to this
  notebook and never shared with the other MVP run notebooks
  (FIX.md F17 -- an earlier version shared one `colab-checkpoints`
  branch across all four, which collided and failed every single
  push under real concurrent load) -- 20 small
  binary commits (one per round, re-run disconnects add more) have no
  business sitting in the same history a reviewer sees when they clone
  this repo to read the paper's actual code.
- `fl/round_loop.py`'s new `on_checkpoint_saved` hook (2026-11-16) keeps
  the FL library itself git-agnostic -- it just calls back with
  `(checkpoint_path, round_idx)` after every local save; all the git
  mechanics below live here, in the notebook, not in the library.

**You'll need a GitHub Personal Access Token** with push access to
`YashasweeNarayan/FL-ZKP` (classic PAT, `repo` scope, or a fine-grained
token scoped to just this repo). The cell below prompts for it via
`getpass` (never echoed to the cell's output, never written to a file
or committed) and deletes it from memory immediately after use.

**On a disconnect and re-run**: this clone lives on local Colab disk,
so it does NOT survive the disconnect (same as the code clone) -- but
everything pushed to this notebook's own `colab-checkpoints-case1-tiny_cnn` before the
disconnect DID survive, on GitHub. Re-running this cell re-clones fresh and fetches
that branch if it already exists, so the run cell's `resume_from` still
finds the latest checkpoint and continues from there, not round 0.


In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
CKPT_BRANCH = "colab-checkpoints-case1-tiny_cnn"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does colab-checkpoints already exist on the remote (pushed in an
    # earlier, since-disconnected session)? if so, use it -- that's where
    # the actual latest checkpoint lives. If not, this is genuinely new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): this notebook has its own dedicated
    CKPT_BRANCH, never shared with the other MVP run notebooks, so a
    collision with another notebook's push is no longer possible by
    construction. The fetch+rebase retry below is kept as defense-in-
    depth only (e.g. if this same notebook is ever accidentally run
    twice at once), retried once before giving up for this round
    (next round's push tries again from scratch regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer -- fetch + rebase
            # onto the remote tip and retry once (see docstring above:
            # this should be rare now that each notebook has its own
            # dedicated branch).
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The real run: 20 rounds, checkpointed every round and pushed to
# this notebook's own dedicated 'colab-checkpoints-case1-tiny_cnn' git branch (see the
# git-checkpoint-setup cell above -- CKPT_DIR/push_checkpoint_to_git
# come from there, not Drive).
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

REAL_ROUNDS = 20  # down from CFG['mvp_scope']['rounds_per_run'] (45) -- see markdown cell 0

# FIX.md F9 (2026-11-15, DP noise scaling fix): any checkpoint from before
# that fix was trained with per-client noise sigma_dp/sqrt(1) instead of
# sigma_dp/sqrt(p*n) -- ~7x too much noise at this run's scale -- and is
# NOT compatible with resuming under the fix (the model's weights, mid-
# training, reflect the old, wrong noise level).
#
# FIX.md F14 (2026-11-16, dummy-client DP noise bug): a second,
# independent invalidation on top of F9 -- the completed 20-round run
# that used the _f9fix.pt checkpoint (pushed 2026-11-16 05:57-10:56 UTC)
# ran entirely BEFORE F14 landed (committed 15:51 UTC that day), so it
# still added DP noise to every one of the ~50 clients instead of just
# the ~12 real (non-dummy) ones. That run's accuracy is invalid for the
# same reason the earlier 33.34%/33.61% run was. New checkpoint filename
# (_f14fix) so this run starts fresh from round 0 under the current,
# correct code rather than silently resuming the pre-F14 weights from
# CKPT_DIR/case1_tiny_cnn_r20_f9fix.pt.
# '_nodp' suffix (FIX.md F19, 2026-11-17): the DP layer is now
# disabled (configs/mvp_3day.yaml, dp.sigma: 0.0). Every checkpoint
# written BEFORE that change was trained with DP noise that measured
# out at 42.4% MNIST / 1.3% FEMNIST accuracy; resuming one would carry
# those noise-corrupted weights into this run. New filename so this
# starts clean from round 0 instead of silently continuing them.
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_r{REAL_ROUNDS}_nodp.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"bootstrap-{REAL_MODEL}".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()
n_accepted = sum(1 for h in real_history if not h['aborted'])
print(f"done: {len(real_history)} rounds in history, {n_accepted} accepted (not aborted)")

## Evaluate accuracy

Works whether you let training finish, stopped it early (Ctrl-C /
interrupt), or are coming back in a fresh runtime with nothing but the
checkpoint pushed to this notebook's own dedicated git branch: it
loads the LATEST checkpointed weights directly
from `CKPT_PATH` (falling back to the in-memory `final_w_t` only if no
checkpoint file exists yet), so it always reports accuracy for
whatever round you actually reached, not a partially-updated `model`
object. Reusable as-is for `small_resnet` later -- every name it reads
(`REAL_MODEL`, `model`, `model_shapes`, `CKPT_PATH`, `case1_data`) is
already generic, so nothing here needs to change when you switch
backbones.


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- this is what makes
# the cell correct even if you stopped training early: `final_w_t`/
# `model` in memory only reflect whatever round your training cell
# actually got through before you interrupted it, but so does the
# checkpoint (it's written after every round), so reading from
# CKPT_PATH and reading from memory agree whenever both exist. Preferring
# the checkpoint means this cell also works standalone after a fresh
# runtime restart, with no retraining needed.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")
